# Sistema de Q&A con Routing entre SQL y Documentos (RAG)

Este notebook implementa un sistema de preguntas y respuestas capaz de responder consultas a partir de dos fuentes de información:

- Una base de datos relacional consultada mediante SQL
- Un conjunto de documentos de texto consultados mediante búsqueda semántica

El sistema utiliza embeddings para recuperar documentos relevantes y un mecanismo de routing para decidir automáticamente si una pregunta debe resolverse mediante SQL o mediante recuperación de documentos.

## 1 - Importo Librerías

In [1]:
import os #para navegar los documentos
import sqlite3 #para importar la base SQL
import pandas as pd #para leer/trabajar los datos SQL
import numpy as np #para calculos

from sentence_transformers import SentenceTransformer #para embeddings
from sklearn.metrics.pairwise import cosine_similarity #para el calculo de similaridad de coseno en los embeddings
import warnings #para ignorar alertas a la hora de ejecutar la notebook
warnings.filterwarnings('ignore')

## 2 - Conexión Base SQL y exploracion de la base de datos.

In [2]:
conn = sqlite3.connect("tienda.db")

2.1 Vemos que tablas hay

In [3]:
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
tables

,name
0,productos
1,clientes
2,pedidos


2.2 Investigo que hay dentro de las tablas

Productos

In [4]:
pd.read_sql("SELECT * FROM productos;", conn)

,id,nombre,categoria,precio,stock
0,1,Laptop HP,Electrónicos,899.99,15
1,2,Mouse Inalámbrico,Electrónicos,25.99,50
2,3,Silla de Oficina,Hogar,199.99,8
3,4,Lámpara LED,Hogar,45.50,30
4,5,Zapatillas Running,Deportes,79.99,25
5,6,Camiseta Deportiva,Deportes,29.99,40
6,7,Jeans Azules,Moda,59.99,20
7,8,Chaqueta de Cuero,Moda,149.99,10
8,9,Teclado Mecánico,Electrónicos,119.99,12
9,10,Auriculares Bluetooth,Electrónicos,89.99,18


*Veo cuántas filas tiene la tabla*

In [5]:
pd.read_sql("SELECT count(*) FROM productos;", conn)

,count(*)
0,10


Clientes

In [6]:
pd.read_sql("SELECT * FROM clientes;", conn)

,id,nombre,email,nivel
0,1,Juan Pérez,juan@email.com,basico
1,2,María García,maria@email.com,premium
2,3,Carlos López,carlos@email.com,basico
3,4,Ana Martínez,ana@email.com,premium
4,5,Luis Rodríguez,luis@email.com,basico


*Veo cuántas filas tiene la tabla*

In [7]:
pd.read_sql("SELECT count(*) FROM clientes;", conn)

,count(*)
0,5


Pedidos

In [8]:
pd.read_sql("SELECT * FROM pedidos;", conn)

,id,cliente_id,fecha,total,estado
0,1,5,2025-10-22,351.94,Procesando
1,2,2,2025-09-29,371.27,En Camino
2,3,3,2025-10-08,451.05,En Camino
3,4,1,2025-10-05,485.12,En Camino
4,5,4,2025-09-30,306.29,Entregado
5,6,1,2025-10-13,83.68,Entregado
6,7,2,2025-10-22,417.47,En Camino
7,8,5,2025-10-19,383.97,Entregado
8,9,3,2025-09-30,160.62,Entregado
9,10,3,2025-10-01,346.95,En Camino


*Veo cuántas filas tiene la tabla*

In [9]:
pd.read_sql("SELECT count(*) as q_pedidos FROM pedidos;", conn)

,q_pedidos
0,20


#### 2.3 - Consultas a las bases

*¿Cuántas compras se hicieron por periodo y cuánto se gastó?*

In [10]:
pd.read_sql("SELECT (substr(fecha,1,4)*100 + substr(fecha,6,2)) as periodo, count(*) as q_pedidos, sum (total) as gasto_total FROM pedidos group by 1 order by periodo;", conn)

,periodo,q_pedidos,gasto_total
0,202509,4,1017.29
1,202510,16,5243.82


En octubre fue el mes que más compras hubo y que más dineró se gastó.

*¿Cuantos pedidos hizo cada cliente y cuánta plata gastó?*

In [11]:
pd.read_sql("SELECT cl.nombre, count(*) as q_pedidos, sum (total) as gasto_total FROM pedidos as pd LEFT JOIN clientes cl on pd.cliente_id = cl.id group by 1 order by q_pedidos desc;", conn)

,nombre,q_pedidos,gasto_total
0,Carlos López,6,2017.42
1,Ana Martínez,5,1586.04
2,María García,4,1160.01
3,Juan Pérez,3,761.73
4,Luis Rodríguez,2,735.91


Carlos López es el cliente que más compró y también el que más gastó.

*¿Cuantos stock tenemos por categoría?*

In [12]:
pd.read_sql("SELECT categoria, sum(stock) as stock FROM productos group by 1 order by stock desc;", conn)

,categoria,stock
0,Electrónicos,95
1,Deportes,65
2,Hogar,38
3,Moda,30


En electrónicos es donde más stock tenemos.

*¿Es electrónica también la categoría en la que mayor dinero hay invertido?*

In [13]:
pd.read_sql("SELECT categoria, sum(stock*precio) as inversion_total FROM productos group by 1 order by inversion_total desc;", conn)

,categoria,inversion_total
0,Electrónicos,17859.05
1,Deportes,3199.35
2,Hogar,2964.92
3,Moda,2699.70


Efectivamente, también es en donde más dinero si invirtió.

## 3 - Cargar documentos de texto

In [14]:
documents = {} #creo un diccionario

folder_path = "/content" #camino en colab

for file in os.listdir(folder_path):
    if file.endswith(".txt"):
        with open(os.path.join(folder_path, file), "r", encoding="utf-8") as f:
            documents[file] = f.read()

for key in documents:
  print(f"{key} archivo cargado")

contacto.txt archivo cargado
devoluciones.txt archivo cargado
envios.txt archivo cargado
productos.txt archivo cargado


*Valido el contenido de los documentos*

In [15]:
for key, value in documents.items():
  print(f"{key}:\n\n{value}")


contacto.txt:

INFORMACIÓN DE CONTACTO

Email: info@tienda.com
Teléfono: +34 900 123 456
Horario: Lunes a Viernes 9:00-18:00
Chat en vivo: disponible en la web

devoluciones.txt:

POLÍTICA DE DEVOLUCIONES

30 días para devolver productos
Producto debe estar sin usar
Reembolso en 5-7 días hábiles
Envío de devolución gratis

envios.txt:

POLÍTICA DE ENVÍOS

Envío gratis en pedidos superiores a €50
Entrega en 3-5 días hábiles
Transportistas: DHL, UPS, Correos
Seguimiento disponible 24h después del envío

productos.txt:

INFORMACIÓN DE PRODUCTOS

Categorías: Electrónica, Muebles, Accesorios
Garantía de 2 años en electrónica
Garantía de 1 año en muebles
Accesorios sin garantía extendida
Soporte técnico: soporte@tienda.com



El contenido se ve correctamente cargado.

*Reviso la cantidad de caracteres por archivo para el embedding.*

In [16]:
for name, text in documents.items():
    print(name, "→", len(text), "caracteres")

contacto.txt → 145 caracteres
devoluciones.txt → 144 caracteres
envios.txt → 168 caracteres
productos.txt → 204 caracteres


## 4 - Crear Embeddings

Para poder realizar búsqueda semántica, convertimos cada documento en un embedding utilizando un modelo de sentence-transformers. Esto permite representar el significado del texto en un espacio vectorial.

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Elijo este modelo por que es chico y rápido para la tarea que tenemos que ejecutar dados los datos que tenemos.

#### *Creo el embedding*

In [18]:
document_embeddings = {}

for name, text in documents.items():
    embedding = model.encode(text)
    document_embeddings[name] = embedding

In [19]:
for keys, values in document_embeddings.items():
  print(f"{keys}: {values[:5]}\n")

contacto.txt: [-0.08353983 -0.0140098   0.02302019 -0.05856885 -0.07072285]

devoluciones.txt: [-0.00653008  0.01069556 -0.03358679 -0.10550167 -0.02589149]

envios.txt: [ 0.00531272  0.03561421  0.01804127 -0.01653695 -0.04547873]

productos.txt: [-0.02070716 -0.01762973 -0.03934735 -0.08348638  0.0265038 ]



Los vectores fueron creados.

Valido el tamaño de los vectores:

In [20]:
for name, emb in document_embeddings.items():
    print(name, "→ tamaño del vector:", len(emb))

contacto.txt → tamaño del vector: 384
devoluciones.txt → tamaño del vector: 384
envios.txt → tamaño del vector: 384
productos.txt → tamaño del vector: 384


## 5 -  Implemento búsqueda semántica

In [21]:
def busqueda_documentos(question):

    question_embedding = model.encode(question)

    best_doc = None
    best_score = -1

    for name, emb in document_embeddings.items():
        score = cosine_similarity(
            [question_embedding],
            [emb]
        )[0][0]

        if score > best_score:
            best_score = score
            best_doc = name

    return best_doc, best_score

Caso de prueba:

In [22]:
busqueda_documentos("cómo me contacto con la tienda?")

('contacto.txt', np.float32(0.7427274))

Mejoramos display:

In [23]:
question = input("Escribe tu pregunta: ")

result = busqueda_documentos(question)

print("\nDocumento encontrado:", result[0])

Escribe tu pregunta: cómo me contacto con la tienda?

Documento encontrado: contacto.txt


In [24]:
while True:

    question = input("\nPregunta (escribe 'salir' para terminar): ")

    if question.lower() == "salir":
        break

    result = busqueda_documentos(question)

    print("\nDocumento encontrado:", result[0])
    print("Similitud:", result[1])


Pregunta (escribe 'salir' para terminar): cuando llega mi producto?

Documento encontrado: devoluciones.txt
Similitud: 0.51283294

Pregunta (escribe 'salir' para terminar): que garantía tiene mi compra?

Documento encontrado: contacto.txt
Similitud: 0.45829147

Pregunta (escribe 'salir' para terminar): que garantía tiene un mueble?

Documento encontrado: contacto.txt
Similitud: 0.364096

Pregunta (escribe 'salir' para terminar): los accesorios tienen garantía?

Documento encontrado: contacto.txt
Similitud: 0.3542275

Pregunta (escribe 'salir' para terminar): cuanto tarda un envío?

Documento encontrado: devoluciones.txt
Similitud: 0.4085191

Pregunta (escribe 'salir' para terminar): quien es el transportista?

Documento encontrado: envios.txt
Similitud: 0.5179711

Pregunta (escribe 'salir' para terminar): en cuanto me reembolsan?

Documento encontrado: devoluciones.txt
Similitud: 0.37298575

Pregunta (escribe 'salir' para terminar): cuanto hay que gastar para un envio gratis?

Documen

Responde algunas consultas bien, pero la gran mayoría mal. En este caso las preguntas respondidas correctamente fueron 2/9 **~22%**.

Voy a probar cambiando de modelo a alguno que esté más relacionado al español ya que el utilizado está optimizado para inglés.

También, para darle algo más de contexto al modelo, voy a agregar los títulos de los documentos al embedding.

In [25]:
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Nuevo modelo corrido.

In [26]:
document_embeddings = {}

for name, text in documents.items():
    enriched_text = f"Documento sobre {name.replace('.txt','')}: {text}"
    embedding = model.encode(enriched_text)
    document_embeddings[name] = embedding

Contexto agregado.

*La función previamente creada para utilizar la similitud de coseno sigue funcionando porque reemplacé los objetos pero no el nombre del objeto.*

In [27]:
chat_history = []

Guardo las preguntas y respuestas en una lista para después comparar. También, directamente creo la función de preguntas/respuestas con su guardado.

In [28]:
def chat_QA():

    while True:

        question = input("\nPregunta (escribe 'salir' para terminar): ")

        if question.lower() == "salir":
            break

        doc, score = busqueda_documentos(question)

        print("\nDocumento encontrado:", doc)
        print("Similitud:", score)

        feedback = input("\n¿La respuesta fue útil? (si/no): ").lower()

        chat_history.append({
            "question": question,
            "document": doc,
            "score": score,
            "feedback": feedback
        })

    return pd.DataFrame(chat_history)['feedback'].value_counts(normalize=True) * 100

In [29]:
chat_QA()


Pregunta (escribe 'salir' para terminar): cuando llega mi producto?

Documento encontrado: devoluciones.txt
Similitud: 0.36113706

¿La respuesta fue útil? (si/no): no

Pregunta (escribe 'salir' para terminar): que garantía tiene mi compra?

Documento encontrado: productos.txt
Similitud: 0.35841084

¿La respuesta fue útil? (si/no): si

Pregunta (escribe 'salir' para terminar): que garantía tiene un mueble?

Documento encontrado: devoluciones.txt
Similitud: 0.2859421

¿La respuesta fue útil? (si/no): no

Pregunta (escribe 'salir' para terminar): los accesorios tienen garantía?

Documento encontrado: productos.txt
Similitud: 0.5521131

¿La respuesta fue útil? (si/no): si

Pregunta (escribe 'salir' para terminar): cuanto tarda un envío?

Documento encontrado: envios.txt
Similitud: 0.5914755

¿La respuesta fue útil? (si/no): si

Pregunta (escribe 'salir' para terminar): quien es el transportista?

Documento encontrado: envios.txt
Similitud: 0.3911909

¿La respuesta fue útil? (si/no): si

P

,proportion
feedback,
si,66.666667
no,33.333333


In [30]:
pd.DataFrame(chat_history)['feedback'].value_counts(normalize=True) * 100

,proportion
feedback,
si,66.666667
no,33.333333


Vemos una amplia mejora al **~67%** con los cambios efectuados. Nos quedamos con el modelo multilingual.

Para evitar tener que escribir las preguntas 1 a 1, dejo también la opción de mandar todas juntas:

In [31]:
questions = [
    "cuando llega mi producto?",
    "que garantía tiene mi compra?",
    "que garantía tiene un mueble?",
    "los accesorios tienen garantía?",
    "cuanto tarda un envío?",
    "quien es el transportista?",
    "en cuanto me reembolsan?",
    "cuanto hay que gastar para un envio gratis?",
    "cuanto tarda la entrega?"
]

In [32]:
resultados = []

for q in questions:

    doc, score = busqueda_documentos(q)

    resultados.append({
        "pregunta": q,
        "documento": doc,
        "score": score
    })

In [33]:
pd.DataFrame(resultados)

,pregunta,documento,score
0,cuando llega mi producto?,devoluciones.txt,0.361137
1,que garantía tiene mi compra?,productos.txt,0.358411
2,que garantía tiene un mueble?,devoluciones.txt,0.285942
3,los accesorios tienen garantía?,productos.txt,0.552113
4,cuanto tarda un envío?,envios.txt,0.591475
5,quien es el transportista?,envios.txt,0.391191
6,en cuanto me reembolsan?,devoluciones.txt,0.616268
7,cuanto hay que gastar para un envio gratis?,envios.txt,0.662624
8,cuanto tarda la entrega?,devoluciones.txt,0.510509


Lo guardamos para futuras mejoras, sobre todo, para validar las preguntas finales en caso de que no sean bien respondidas.

 ## 6 - Sistema de Q&A (routing) + Preguntas de prueba

Voy a hacer dos versiones, una "barata" con palabras más relacionadas a SQL y otro donde todas las preguntas pasan por el proceso de embedding, más "caro"/"costoso" computacionalmente, pero con mejor experiencia para el usuario.

#### 6.1 Modelo Simple

Ejemplos de palabras para sql

In [34]:
palabras_sql = [
    "cuántos",
    "cuantas",
    "cantidad",
    "total",
    "clientes",
    "pedidos",
    "compras",
    "stock",
    "categoría",
    "ventas",
    "gastó",
    "gasto"
]

Armo enrutador

In [35]:
def enrutador_preguntas(pregunta):

    q = pregunta.lower()

    for word in palabras_sql:
        if word in q:
            return "sql"

    return "documentos"

Función para preguntas relacionadas a SQL

In [36]:
def respuesta_sql(pregunta):

    if "clientes" in pregunta.lower():
        query = "SELECT COUNT(*) FROM clientes"
        result = pd.read_sql(query, conn)
        respuesta = f"Tenemos {result.iloc[0,0]} clientes."

    elif "stock por categoria" in pregunta.lower():
        query = """
        SELECT categoria, SUM(stock) as stock_total
        FROM productos
        GROUP BY 1
        """
        result = pd.read_sql(query, conn)
        respuesta = result.to_string(index=False)

    elif "cuantos pedidos" in pregunta.lower() or "cuántos pedidos" in pregunta.lower():
        query = """
        SELECT count(distinct id) as pedidos_totales FROM pedidos
        """
        result = pd.read_sql(query, conn)
        respuesta = f"Tenemos {result.iloc[0,0]} pedidos."

    else:
        respuesta = "No tengo una consulta SQL definida para esa pregunta."

    return respuesta, "SQL"

Función para preguntas relacionadas a documentos

In [37]:
def respuesta_documentos(pregunta):

    doc, score = busqueda_documentos(pregunta)

    respuesta = documents.get(doc, "")

    return respuesta, doc

Sistema Final QA:

In [38]:
def sistema_QA(pregunta):

    route = enrutador_preguntas(pregunta)

    if route == "sql":
        answer, source = respuesta_sql(pregunta)

    else:
        answer, source = respuesta_documentos(pregunta)

    return {
        "pregunta": pregunta,
        "respuesta": answer,
        "fuente": source
    }

Pruebas con preguntas de ejemplo

In [39]:
sistema_QA("Cuántos clientes tenemos?")

{'pregunta': 'Cuántos clientes tenemos?',
 'respuesta': 'Tenemos 5 clientes.',
 'fuente': 'SQL'}

In [40]:
sistema_QA("Cuánto cuesta el envío nacional?")

{'pregunta': 'Cuánto cuesta el envío nacional?',
 'respuesta': 'POLÍTICA DE ENVÍOS\n\nEnvío gratis en pedidos superiores a €50\nEntrega en 3-5 días hábiles\nTransportistas: DHL, UPS, Correos\nSeguimiento disponible 24h después del envío\n',
 'fuente': 'envios.txt'}

In [41]:
sistema_QA("Cuantos pedidos hay en total?")

{'pregunta': 'Cuantos pedidos hay en total?',
 'respuesta': 'Tenemos 20 pedidos.',
 'fuente': 'SQL'}

In [42]:
sistema_QA("Qué garantía tienen los productos?")

{'pregunta': 'Qué garantía tienen los productos?',
 'respuesta': 'INFORMACIÓN DE PRODUCTOS\n\nCategorías: Electrónica, Muebles, Accesorios\nGarantía de 2 años en electrónica\nGarantía de 1 año en muebles\nAccesorios sin garantía extendida\nSoporte técnico: soporte@tienda.com\n',
 'fuente': 'productos.txt'}

Guardo todo en una lista para hacer pruebas y luego poder verlas de mejor manera

In [43]:
chat_history_v2 = []

In [44]:
def chat_QA_v2():

    while True:

      pregunta = input("\nPregunta (escribe 'salir'): ")

      if pregunta.lower() == "salir":
          break

      resultado = sistema_QA(pregunta)

      print("\nRespuesta:")
      print(resultado["respuesta"])

      print("\nFuente:", resultado["fuente"])

      feedback = input("\n¿La respuesta fue útil? (si/no): ").lower()

      chat_history_v2.append({
            "pregunta": pregunta,
            "respuesta": resultado["respuesta"],
            "fuente":resultado["fuente"],
            "feedback": feedback
        })

Armamos un chatbot para iterar las preguntas y respuestas con feedback

In [45]:
chat_QA_v2()


Pregunta (escribe 'salir'): ¿Cuánto cuesta el envío nacional?

Respuesta:
POLÍTICA DE ENVÍOS

Envío gratis en pedidos superiores a €50
Entrega en 3-5 días hábiles
Transportistas: DHL, UPS, Correos
Seguimiento disponible 24h después del envío


Fuente: envios.txt

¿La respuesta fue útil? (si/no): si

Pregunta (escribe 'salir'): ¿Cuántos pedidos hay en total?

Respuesta:
Tenemos 20 pedidos.

Fuente: SQL

¿La respuesta fue útil? (si/no): si

Pregunta (escribe 'salir'): ¿Puedo devolver un producto?

Respuesta:
POLÍTICA DE DEVOLUCIONES

30 días para devolver productos
Producto debe estar sin usar
Reembolso en 5-7 días hábiles
Envío de devolución gratis


Fuente: devoluciones.txt

¿La respuesta fue útil? (si/no): si

Pregunta (escribe 'salir'): ¿Cuántos clientes tenemos?

Respuesta:
Tenemos 5 clientes.

Fuente: SQL

¿La respuesta fue útil? (si/no): si

Pregunta (escribe 'salir'): ¿Qué garantía tienen los productos?

Respuesta:
INFORMACIÓN DE PRODUCTOS

Categorías: Electrónica, Muebles, Acce

Vemos que las preguntas y respuestas hechas son correctamente canalizadas y respondidas.

#### 6.2 Modelo Sofisticado

Ejemplos de preguntas para preparar el modelo

In [46]:
ejemplos_sql = [
    "¿Cuántos clientes tenemos?",
    "¿Cuántos pedidos se hicieron?",
    "¿Cuál es el total de ventas?",
    "¿Cuánto stock hay por categoría?",
    "¿Cuánto gastó cada cliente?"
]

ejemplos_doc = [
    "¿Cómo puedo devolver un producto?",
    "¿Cuánto tarda el envío?",
    "¿Cómo contacto al soporte?",
    "¿Cuál es la política de devoluciones?",
    "¿Quién realiza el envío?",
    "¿Qué garantía tienen los productos?"
]

Preparo embeddings de los ejemplos

In [47]:
sql_embeddings = model.encode(ejemplos_sql)
doc_embeddings = model.encode(ejemplos_doc)

Enrutador con pregunta con embedding

In [48]:
def enrutador_con_embedding(pregunta):

    p_emb = model.encode(pregunta)

    sql_score = cosine_similarity([p_emb], sql_embeddings).max()
    doc_score = cosine_similarity([p_emb], doc_embeddings).max()

    if sql_score > doc_score:
        return "SQL"
    else:
        return "Documents"

Pruebo el enrutador

In [49]:
enrutador_con_embedding("¿Cuántos clientes tenemos?")

'SQL'

In [50]:
enrutador_con_embedding("¿Qué garantía tienen los productos?")

'Documents'

In [51]:
enrutador_con_embedding("¿Qué garantía tienen los muebles?")

'Documents'

In [52]:
enrutador_con_embedding("¿Hay algún chat de contacto?")

'Documents'

funciona OK

In [53]:
def sistema_QA_v2(pregunta):

    ruta = enrutador_con_embedding(pregunta)

    if ruta == "SQL":
        respuesta, fuente = respuesta_sql(pregunta)

    else:
        respuesta, fuente = respuesta_documentos(pregunta)

    return {
        "pregunta": pregunta,
        "respuesta": respuesta,
        "fuente": fuente,
        "ruta": ruta
    }

Pruebas

In [54]:
sistema_QA_v2("Cuántos clientes tenemos?")

{'pregunta': 'Cuántos clientes tenemos?',
 'respuesta': 'Tenemos 5 clientes.',
 'fuente': 'SQL',
 'ruta': 'SQL'}

In [55]:
sistema_QA_v2("Cuánto cuesta el envío nacional?")

{'pregunta': 'Cuánto cuesta el envío nacional?',
 'respuesta': 'POLÍTICA DE ENVÍOS\n\nEnvío gratis en pedidos superiores a €50\nEntrega en 3-5 días hábiles\nTransportistas: DHL, UPS, Correos\nSeguimiento disponible 24h después del envío\n',
 'fuente': 'envios.txt',
 'ruta': 'Documents'}

In [56]:
sistema_QA_v2("Cuantos pedidos hay en total?")

{'pregunta': 'Cuantos pedidos hay en total?',
 'respuesta': 'Tenemos 20 pedidos.',
 'fuente': 'SQL',
 'ruta': 'SQL'}

In [57]:
sistema_QA_v2("Qué garantía tienen los productos?")

{'pregunta': 'Qué garantía tienen los productos?',
 'respuesta': 'INFORMACIÓN DE PRODUCTOS\n\nCategorías: Electrónica, Muebles, Accesorios\nGarantía de 2 años en electrónica\nGarantía de 1 año en muebles\nAccesorios sin garantía extendida\nSoporte técnico: soporte@tienda.com\n',
 'fuente': 'productos.txt',
 'ruta': 'Documents'}

Guardo todo en una lista para hacer pruebas y luego poder verlas de mejor manera

In [58]:
chat_history_v3 = []

In [59]:
def chat_QA_v3():
    while True:

      pregunta = input("\nPregunta (escribe 'salir'): ")

      if pregunta.lower() == "salir":
          break

      resultado = sistema_QA_v2(pregunta)

      print("\nRespuesta:")
      print(resultado["respuesta"])

      print("\nFuente:", resultado["fuente"])
      print("\nRuta:", resultado["ruta"])

      feedback = input("\n¿La respuesta fue útil? (si/no): ").lower()

      chat_history_v3.append({
            "pregunta": pregunta,
            "respuesta": resultado["respuesta"],
            "fuente":resultado["fuente"],
            "ruta": resultado["ruta"],
            "feedback": feedback
        })

Armamos un chatbot para iterar las preguntas y respuestas con feedback

In [60]:
chat_QA_v3()


Pregunta (escribe 'salir'): ¿Cuánto cuesta el envío nacional?

Respuesta:
POLÍTICA DE ENVÍOS

Envío gratis en pedidos superiores a €50
Entrega en 3-5 días hábiles
Transportistas: DHL, UPS, Correos
Seguimiento disponible 24h después del envío


Fuente: envios.txt

Ruta: Documents

¿La respuesta fue útil? (si/no): si

Pregunta (escribe 'salir'): ¿Cuántos pedidos hay en total?

Respuesta:
Tenemos 20 pedidos.

Fuente: SQL

Ruta: SQL

¿La respuesta fue útil? (si/no): si

Pregunta (escribe 'salir'): ¿Puedo devolver un producto?

Respuesta:
POLÍTICA DE DEVOLUCIONES

30 días para devolver productos
Producto debe estar sin usar
Reembolso en 5-7 días hábiles
Envío de devolución gratis


Fuente: devoluciones.txt

Ruta: Documents

¿La respuesta fue útil? (si/no): si

Pregunta (escribe 'salir'): ¿Cuántos clientes tenemos?

Respuesta:
Tenemos 5 clientes.

Fuente: SQL

Ruta: SQL

¿La respuesta fue útil? (si/no): si

Pregunta (escribe 'salir'): ¿Qué garantía tienen los productos?

Respuesta:
INFORMAC

Dependiendo las necesidades de negocio que tengamos, tal vez este modelo no es tan necesario ya que el anterior responde bien las consultas.

De todas formas, las preguntas realizadas, ya están contempladas previamente por la base de SQL. Si se busca responder algo por fuera de eso, sería mejor mantener un modelo como este y/o evolucionarlo a alguno que permita transformar las preguntas de lenguaje natural en sintáxis SQL.


----------------------------------------------------------------------------

Más allá de este pequeño comentario, este modelo responde correctamente todas las preguntas. Pero me gustaría ir un poco más allá y que responda solamente la parte del documento relacionada a la pregunta. Para eso vamos a utilizar chunks en los documentos (.txt), y en lugar de recuperar el documento completo, el sistema realiza la búsqueda semántica sobre estos chunks y devuelve el más relevante para la pregunta del usuario.

#### 6.3 Modelo con Chunks

Dividimos al documento en chunks (fragmentos):

In [61]:
chunks = []
chunk_docs = []

for name, text in documents.items():

    oraciones = text.split("\n") #usamos el salto de línea por como están hechos los documentos

    for s in oraciones:
        chunks.append(s)
        chunk_docs.append(name)

Veo como quedan los chunks

In [62]:
chunks

['INFORMACIÓN DE CONTACTO',
 '',
 'Email: info@tienda.com',
 'Teléfono: +34 900 123 456',
 'Horario: Lunes a Viernes 9:00-18:00',
 'Chat en vivo: disponible en la web',
 '',
 'POLÍTICA DE DEVOLUCIONES',
 '',
 '30 días para devolver productos',
 'Producto debe estar sin usar',
 'Reembolso en 5-7 días hábiles',
 'Envío de devolución gratis',
 '',
 'POLÍTICA DE ENVÍOS',
 '',
 'Envío gratis en pedidos superiores a €50',
 'Entrega en 3-5 días hábiles',
 'Transportistas: DHL, UPS, Correos',
 'Seguimiento disponible 24h después del envío',
 '',
 'INFORMACIÓN DE PRODUCTOS',
 '',
 'Categorías: Electrónica, Muebles, Accesorios',
 'Garantía de 2 años en electrónica',
 'Garantía de 1 año en muebles',
 'Accesorios sin garantía extendida',
 'Soporte técnico: soporte@tienda.com',
 '']

Creo embeddings de los chunks

In [63]:
chunk_embeddings = model.encode(chunks)

Ahora cada frase tiene su vector asociado

Busco el fragmento más relevante

In [64]:
def search_chunks(pregunta):

    q_emb = model.encode(pregunta)

    scores = cosine_similarity([q_emb], chunk_embeddings)[0]

    best_idx = scores.argmax()

    return chunks[best_idx], chunk_docs[best_idx], scores[best_idx]

Probamos el modelo:

In [65]:
chunk, source, score = search_chunks("¿Cuánto tarda el envío?")

print("Respuesta:", chunk)
print("Fuente:", source)

Respuesta: Seguimiento disponible 24h después del envío
Fuente: envios.txt


In [66]:
chunk, source, score = search_chunks("¿Cuánto cuesta el envío nacional?")

print("Respuesta:", chunk)
print("Fuente:", source)

Respuesta: Envío gratis en pedidos superiores a €50
Fuente: envios.txt


In [67]:
chunk, source, score = search_chunks("¿Puedo devolver un producto?")

print("Respuesta:", chunk)
print("Fuente:", source)

Respuesta: 30 días para devolver productos
Fuente: devoluciones.txt


In [68]:
chunk, source, score = search_chunks("¿Qué garantía tienen los productos?")

print("Respuesta:", chunk)
print("Fuente:", source)

Respuesta: Producto debe estar sin usar
Fuente: devoluciones.txt


Algunas respuestas están bien, pero a otras se les nota que les falta contexto.

Vamos a rehacer el algoritmo con el nombre del documento + fragmentos de al menos 60 caracteres. Elijo esa cantidad de caracteres en base al conteo del punto 4 cuando hicimos los embeddings sobre los documentos.

In [69]:
chunks = []
chunk_docs = []

tamanio_chunk = 60

for name, text in documents.items():

    for i in range(0, len(text), tamanio_chunk):

        chunk = text[i:i+tamanio_chunk]

        # contexto agregado
        enriched_chunk = f"Documento sobre {name.replace('.txt','')}: {chunk}"

        chunks.append(enriched_chunk)
        chunk_docs.append(name)

Vemos como queda el chunk:

In [70]:
chunks

['Documento sobre contacto: INFORMACIÓN DE CONTACTO\n\nEmail: info@tienda.com\nTeléfono: +3',
 'Documento sobre contacto: 4 900 123 456\nHorario: Lunes a Viernes 9:00-18:00\nChat en vi',
 'Documento sobre contacto: vo: disponible en la web\n',
 'Documento sobre devoluciones: POLÍTICA DE DEVOLUCIONES\n\n30 días para devolver productos\nPr',
 'Documento sobre devoluciones: oducto debe estar sin usar\nReembolso en 5-7 días hábiles\nEnv',
 'Documento sobre devoluciones: ío de devolución gratis\n',
 'Documento sobre envios: POLÍTICA DE ENVÍOS\n\nEnvío gratis en pedidos superiores a €50',
 'Documento sobre envios: \nEntrega en 3-5 días hábiles\nTransportistas: DHL, UPS, Corre',
 'Documento sobre envios: os\nSeguimiento disponible 24h después del envío\n',
 'Documento sobre productos: INFORMACIÓN DE PRODUCTOS\n\nCategorías: Electrónica, Muebles, ',
 'Documento sobre productos: Accesorios\nGarantía de 2 años en electrónica\nGarantía de 1 a',
 'Documento sobre productos: ño en muebles\nAccesori

Volvemos a hacer el modelo

In [71]:
chunk_embeddings = model.encode(chunks)

In [72]:
def search_chunks(question):

    q_emb = model.encode(question)

    scores = cosine_similarity([q_emb], chunk_embeddings)[0]

    best_idx = scores.argmax()

    return chunks[best_idx], chunk_docs[best_idx], scores[best_idx]

Limpiamos respuesta y probamos

In [73]:
chunk, source, score = search_chunks("¿Cuánto cuesta el envío nacional?")

clean_answer = chunk.split(":", 1)[1].strip()

print("Respuesta:", clean_answer)
print("Fuente:", source)
print("Similitud:", score)

Respuesta: Entrega en 3-5 días hábiles
Transportistas: DHL, UPS, Corre
Fuente: envios.txt
Similitud: 0.4524188


In [74]:
chunk, source, score = search_chunks("¿Puedo devolver un producto?")

clean_answer = chunk.split(":", 1)[1].strip()

print("Respuesta:", clean_answer)
print("Fuente:", source)
print("Similitud:", score)

Respuesta: POLÍTICA DE DEVOLUCIONES

30 días para devolver productos
Pr
Fuente: devoluciones.txt
Similitud: 0.50115716


In [75]:
chunk, source, score = search_chunks("¿Qué garantía tienen los productos?")

clean_answer = chunk.split(":", 1)[1].strip()

print("Respuesta:", clean_answer)
print("Fuente:", source)
print("Similitud:", score)

Respuesta: Accesorios
Garantía de 2 años en electrónica
Garantía de 1 a
Fuente: productos.txt
Similitud: 0.6656198


- Vemos que si bien mejora el contexto, las respuestas no son del todo precisas y hasta algunas quedan cortadas.

- Entendemos que la experiencia sería mejor si se pule el modelo, pero el limitante de que por cada pregunta haya que limitarse a "x" caracteres, hace que la tecnología chunk no sea tan performante bajo las características de este ejercicio.

Hacemos un último intento, **chunkeando por 3 oraciones seguidas** (en este caso por salto de línea) y no por caracteres, para no perder el significado.

In [76]:
chunks = []
chunk_docs = []

for name, text in documents.items():

    sentences = text.split("\n")

    for i in range(0, len(sentences), 2): #línea de 3 oraciones seguidas

      chunk = " ".join(sentences[i:i+2])

      enriched_chunk = f"Documento sobre {name.replace('.txt','')}: {chunk}"

      chunks.append(enriched_chunk)
      chunk_docs.append(name)

Veo como quedan los chunks:

In [77]:
chunks

['Documento sobre contacto: INFORMACIÓN DE CONTACTO ',
 'Documento sobre contacto: Email: info@tienda.com Teléfono: +34 900 123 456',
 'Documento sobre contacto: Horario: Lunes a Viernes 9:00-18:00 Chat en vivo: disponible en la web',
 'Documento sobre contacto: ',
 'Documento sobre devoluciones: POLÍTICA DE DEVOLUCIONES ',
 'Documento sobre devoluciones: 30 días para devolver productos Producto debe estar sin usar',
 'Documento sobre devoluciones: Reembolso en 5-7 días hábiles Envío de devolución gratis',
 'Documento sobre devoluciones: ',
 'Documento sobre envios: POLÍTICA DE ENVÍOS ',
 'Documento sobre envios: Envío gratis en pedidos superiores a €50 Entrega en 3-5 días hábiles',
 'Documento sobre envios: Transportistas: DHL, UPS, Correos Seguimiento disponible 24h después del envío',
 'Documento sobre envios: ',
 'Documento sobre productos: INFORMACIÓN DE PRODUCTOS ',
 'Documento sobre productos: Categorías: Electrónica, Muebles, Accesorios Garantía de 2 años en electrónica',
 'Doc

Reproceso el modelo con las nuevas condiciones

In [78]:
chunk_embeddings = model.encode(chunks)

Valido que las tres partes tengan los mismos chunks.

In [79]:
print("chunks:", len(chunks))
print("chunk_docs:", len(chunk_docs))
print("embeddings:", len(chunk_embeddings))

chunks: 16
chunk_docs: 16
embeddings: 16


Reproceso la función

In [80]:
def search_chunks(pregunta):

    q_emb = model.encode(pregunta)

    scores = cosine_similarity([q_emb], chunk_embeddings)[0]

    best_idx = scores.argmax()

    return chunks[best_idx], chunk_docs[best_idx], scores[best_idx]

Limpiamos respuesta y probamos

In [81]:
chunk, source, score = search_chunks("¿Puedo devolver un producto?")

clean_answer = chunk.split(":", 1)[1].strip()

print("Respuesta:", clean_answer)
print("Fuente:", source)
print("Similitud:", score)

Respuesta: 30 días para devolver productos Producto debe estar sin usar
Fuente: devoluciones.txt
Similitud: 0.6774361


Generamos el sistema para probar con las tres preguntas que deberían ser respondidas por esta parte del RAG:

In [82]:
questions = [
    "¿Cuánto cuesta el envío nacional?",
    "¿Puedo devolver un producto?",
    "¿Qué garantía tienen los productos?"
]

for q in questions:
    chunk, source, score = search_chunks(q)

    clean_answer = chunk.split(":", 1)[1].strip()

    print("\nPregunta:", q)
    print("Respuesta:", clean_answer)
    print("Fuente:", source)
    print("Similitud:", score)


Pregunta: ¿Cuánto cuesta el envío nacional?
Respuesta: Transportistas: DHL, UPS, Correos Seguimiento disponible 24h después del envío
Fuente: envios.txt
Similitud: 0.4725653

Pregunta: ¿Puedo devolver un producto?
Respuesta: 30 días para devolver productos Producto debe estar sin usar
Fuente: devoluciones.txt
Similitud: 0.6774361

Pregunta: ¿Qué garantía tienen los productos?
Respuesta: INFORMACIÓN DE PRODUCTOS
Fuente: productos.txt
Similitud: 0.62685573


- No termina de responder bien el modelo a pesar de la iteración. Aunque la similitud de coseno incrementó.

Con los resultados expuestos, vamos a quedarnos con el modelo del punto 6.2, que es el modelo con mayores posibilidades de mejorar.

## 7- Conclusión

En este ejercicio se desarrolló un sistema de **preguntas y respuestas (Q&A)** capaz de responder consultas utilizando dos fuentes de información diferentes: una **base de datos estructurada** y un **conjunto de documentos de texto**.

El flujo parte de la carga y limpieza de los documentos, luego se generan embeddings y finalmente se implementa un sistema que recupera el contenido más relevante para responder preguntas del usuario.

Se exploraron distintas aproximaciones de recuperación. La versión final utiliza embeddings sobre documentos completos, lo que permitió obtener respuestas más estables y mantener un pipeline claro:
pregunta → embedding → búsqueda por similitud → generación de respuesta con contexto.

También se experimentó con chunking de documentos para mejorar la granularidad de la recuperación. Sin embargo, en las pruebas realizadas esta estrategia no produjo respuestas suficientemente coherentes, por lo que no se incorporó en la versión final del sistema.

Como posibles mejoras futuras, se podrían considerar:

- utilizar un modelo de clasificación más robusto para el routing

- incorporar un modelo generativo (LLM) para producir respuestas más naturales

- almacenar los embeddings en una base de datos vectorial para mejorar la escalabilidad

- incorporar un text-to-SQL que traduzca lenguaje natural a queries reales automáticamente.